Human Astrocytes Exhibit Tumor Microenvironment-, Age-, and Sex-Related Transcriptomic Signatures
* https://www.jneurosci.org/content/42/8/1587

Here focues on peritumor AC vs. normal AC

# Import packages

In [1]:
import pandas as pd
import numpy as np

import plotly.express as px

import os

# Read input file

In [2]:
direct_input = 'input/JNEUROSCI.0407-21.2021/DGE_peritumor_AC.xlsx'


In [3]:
df_input = pd.read_excel(direct_input, header = 1, index_col=0)
df_input

,Gene Name,log2(Fold Change),adjusted p,biotype
Ensembl Gene ID,,,,
ENSG00000132688,NES,3.256513,1.180574e-08,protein_coding
ENSG00000117395,EBNA1BP2,1.044100,1.841282e-07,protein_coding
ENSG00000164032,H2AFZ,1.336122,1.841282e-07,protein_coding
ENSG00000102265,TIMP1,2.135396,1.889208e-07,protein_coding
ENSG00000122694,GLIPR2,2.125970,1.889208e-07,protein_coding
...,...,...,...,...
ENSG00000242485,MRPL20,0.511635,4.937106e-02,protein_coding
ENSG00000278983,AC048380.2,-1.106321,4.937106e-02,TEC
ENSG00000168397,ATG4B,-0.454989,4.938938e-02,protein_coding


In [4]:
df_input.columns

Index(['Gene Name', 'log2(Fold Change)', 'adjusted p', 'biotype'], dtype='object')

In [5]:
df_input[df_input['adjusted p'] <= 0.05] #Filter for genes with significant changes
df_input = df_input.reset_index()
df_input

,Ensembl Gene ID,Gene Name,log2(Fold Change),adjusted p,biotype
0,ENSG00000132688,NES,3.256513,1.180574e-08,protein_coding
1,ENSG00000117395,EBNA1BP2,1.044100,1.841282e-07,protein_coding
2,ENSG00000164032,H2AFZ,1.336122,1.841282e-07,protein_coding
3,ENSG00000102265,TIMP1,2.135396,1.889208e-07,protein_coding
4,ENSG00000122694,GLIPR2,2.125970,1.889208e-07,protein_coding
...,...,...,...,...,...
3115,ENSG00000242485,MRPL20,0.511635,4.937106e-02,protein_coding
3116,ENSG00000278983,AC048380.2,-1.106321,4.937106e-02,TEC
3117,ENSG00000168397,ATG4B,-0.454989,4.938938e-02,protein_coding
3118,ENSG00000226986,AC092017.1,1.292896,4.941602e-02,processed_pseudogene


# Process data

In [6]:
df_input['direction'] = np.where(
    df_input['log2(Fold Change)'] >= 0,
    'Upregulated',
    'Downregulated'
)

df_input

,Ensembl Gene ID,Gene Name,log2(Fold Change),adjusted p,biotype,direction
0,ENSG00000132688,NES,3.256513,1.180574e-08,protein_coding,Upregulated
1,ENSG00000117395,EBNA1BP2,1.044100,1.841282e-07,protein_coding,Upregulated
2,ENSG00000164032,H2AFZ,1.336122,1.841282e-07,protein_coding,Upregulated
3,ENSG00000102265,TIMP1,2.135396,1.889208e-07,protein_coding,Upregulated
4,ENSG00000122694,GLIPR2,2.125970,1.889208e-07,protein_coding,Upregulated
...,...,...,...,...,...,...
3115,ENSG00000242485,MRPL20,0.511635,4.937106e-02,protein_coding,Upregulated
3116,ENSG00000278983,AC048380.2,-1.106321,4.937106e-02,TEC,Downregulated
3117,ENSG00000168397,ATG4B,-0.454989,4.938938e-02,protein_coding,Downregulated
3118,ENSG00000226986,AC092017.1,1.292896,4.941602e-02,processed_pseudogene,Upregulated


In [7]:
df_up = df_input[df_input['direction'] == 'Upregulated']
df_down = df_input[df_input['direction'] == 'Downregulated']

direct_output= 'output/JNEUROSCI.0407-21.2021'
os.makedirs(direct_output, exist_ok=True)

for key, df in {'upregulated':df_up, 'downregulated':df_down}.items():
    df = df.sort_values(
        by='log2(Fold Change)',
        key=abs, #Use the absolute value
        ascending=False #Descending
    )
    df = df.reset_index(drop=True)

    direct_save = os.path.join(direct_output, f'DGE_peritumor_AC_{key}.csv')
    df.to_csv(direct_save,index=False)

In [8]:
df_input['-log(adjusted p)'] = -np.log10(df_input['adjusted p'])
df_input

,Ensembl Gene ID,Gene Name,log2(Fold Change),adjusted p,biotype,direction,-log(adjusted p)
0,ENSG00000132688,NES,3.256513,1.180574e-08,protein_coding,Upregulated,7.927907
1,ENSG00000117395,EBNA1BP2,1.044100,1.841282e-07,protein_coding,Upregulated,6.734880
2,ENSG00000164032,H2AFZ,1.336122,1.841282e-07,protein_coding,Upregulated,6.734880
3,ENSG00000102265,TIMP1,2.135396,1.889208e-07,protein_coding,Upregulated,6.723720
4,ENSG00000122694,GLIPR2,2.125970,1.889208e-07,protein_coding,Upregulated,6.723720
...,...,...,...,...,...,...,...
3115,ENSG00000242485,MRPL20,0.511635,4.937106e-02,protein_coding,Upregulated,1.306528
3116,ENSG00000278983,AC048380.2,-1.106321,4.937106e-02,TEC,Downregulated,1.306528
3117,ENSG00000168397,ATG4B,-0.454989,4.938938e-02,protein_coding,Downregulated,1.306366
3118,ENSG00000226986,AC092017.1,1.292896,4.941602e-02,processed_pseudogene,Upregulated,1.306132


# Plotting

In [9]:
fig = px.scatter(
    df_input,
    x='log2(Fold Change)',
    y='-log(adjusted p)',
    hover_name='Gene Name',
    color='direction',
    color_discrete_map={
        'Upregulated': 'red',
        'Downregulated': 'blue'
    },
    title='Differential Gene Expression in Peritumor Astrocytes'
)

fig.update_layout(
    xaxis_title='log2(Fold Change)',
    yaxis_title='-log10(Adjusted p-value)'
)

fig.add_annotation(
    x=0.5,
    y=0,
    text='All genes with adjusted p-value <= 0.05',
    showarrow=False
)

fig.show()

In [10]:
fig.write_html(os.path.join(direct_output,"volcano_plot.html"))